# Notebook 6: Real-World Microphone Audio Recording & In-Jupyter Playback

This notebook demonstrates how to record live sound from **MAX4466 / MAX9814 electret microphones** connected to the PYNQ-Z2 board using the multi-second continuous DMA engine.

### Hardware Wiring:
| Pin | PYNQ-Z2 Connection | Description |
| :--- | :--- | :--- |
| **VCC** | **3.3V** | Power Header |
| **GND** | **GND** | Power Header |
| **OUT (Mic 1)** | **Header J1 Pin A0** | Channel 1 (1.65V DC bias) |
| **OUT (Mic 2)** | **Header J1 Pin A1** | Channel 2 (1.65V DC bias) |

*(No Analog Discovery 3 required — you can speak, whistle, or play music into the microphones!)*

## 1. Initialize Overlay in Full-Band Audio Mode
Loads the overlay and selects the `audio` regime (50 kSPS sampling rate, 25 kHz Nyquist bandwidth).

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

check_usb_permissions()

# Load overlay and switch to 50 kSPS Audio Profile
ol = OscilloscopeOverlay()
ol.set_profile("audio")
print(f"✅ Active Profile: {ol.current_profile} ({ol.sample_rate_hz/1e3:.1f} kSPS)")
print(f"   • Channel 1 : Arduino Header Pin A0 (Mic 1)")
print(f"   • Channel 2 : Arduino Header Pin A1 (Mic 2)")

## 2. Record 3 Seconds of Real Microphone Audio
Execute the cell below and **speak, clap, or play audio near Microphone 1 (A0)**.

In [ ]:
rec_duration = 3.0 # seconds
print(f"🎙️ RECORDING NOW ({rec_duration:.1f} seconds)... Speak or make sound near Mic 1 (A0)!")

t_start = time.time()
# Record Channel 1 (A0) with auto-gain scaling
mic1_audio = ol.record_audio(duration_sec=rec_duration, channel=1, auto_gain=True)
elapsed = time.time() - t_start

print(f"✅ Recording Complete in {elapsed:.2f} s!")
print(f"   • Captured Samples : {len(mic1_audio):,}")
print(f"   • Dynamic Range    : Min = {mic1_audio.min():.3f}, Max = {mic1_audio.max():.3f}")

## 3. Listen to the Recorded Sound in Jupyter
Plays back the recorded microphone audio directly in your browser using the HTML5 audio widget.

In [ ]:
print("🔊 Playing back recorded audio from Microphone 1 (A0):")
ol.play_audio(custom_data=mic1_audio, sample_rate_hz=ol.sample_rate_hz)

## 4. Plot Recorded Acoustic Waveform
Displays the recorded multi-second acoustic waveform.

In [ ]:
time_axis_s = np.linspace(0, rec_duration, len(mic1_audio))

fig_wave = go.Figure()
fig_wave.add_scatter(
    x=time_axis_s, y=mic1_audio,
    mode="lines",
    line=dict(color="#00FFCC", width=1.2),
    name="Mic 1 (A0)"
)
fig_wave.update_layout(
    title=f"<b>Captured Microphone Waveform ({rec_duration:.1f} Seconds @ 50 kSPS)</b>",
    template="plotly_dark",
    xaxis_title="Time (Seconds)",
    yaxis_title="Normalized Audio Amplitude [-1.0, 1.0]",
    height=380
)
fig_wave.show()

## 5. Microscopic Frame-Boundary Continuity Check
Zooms in on the exact sample index ($n = 1024$) where the DMA double-buffer swaps between Frame 1 and Frame 2 to verify **zero dropped samples or glitches**.

In [ ]:
boundary_idx = 1024
zoom_win = 100

slice_idx = np.arange(boundary_idx - zoom_win, boundary_idx + zoom_win)
slice_time_ms = (slice_idx / ol.sample_rate_hz) * 1e3
slice_audio = mic1_audio[slice_idx]

fig_zoom = go.Figure()
fig_zoom.add_scatter(x=slice_time_ms, y=slice_audio, mode="lines+markers", marker=dict(size=4), line=dict(color="#00FFCC", width=2), name="Audio Signal")
fig_zoom.add_vline(x=(boundary_idx / ol.sample_rate_hz) * 1e3, line=dict(color="#FF007F", width=2, dash="dash"), annotation_text="DMA Buffer Boundary (1024)")

fig_zoom.update_layout(
    title="<b>Frame Boundary Transition: Smooth Sinusoidal Continuity</b>",
    template="plotly_dark",
    xaxis_title="Time (ms)",
    yaxis_title="Normalized Amplitude",
    height=350
)
fig_zoom.show()

## 6. Simultaneous Stereo Microphone Recording (Mic 1 & Mic 2)
Records from **both microphones simultaneously** (`channel=0` returns an $[N, 2]$ array for Left/Right audio).

In [ ]:
print("🎙️ Recording 2.0 seconds of STEREO audio from Mic 1 (A0) and Mic 2 (A1)...")
stereo_audio = ol.record_audio(duration_sec=2.0, channel=0, auto_gain=True)

t_stereo = np.linspace(0, 2.0, len(stereo_audio))
show_len = 2000 # Show first 40 ms

fig_stereo = make_subplots(rows=2, cols=1, subplot_titles=("<b>Microphone 1 (Header Pin A0)</b>", "<b>Microphone 2 (Header Pin A1)</b>"))
fig_stereo.add_scatter(x=t_stereo[:show_len]*1e3, y=stereo_audio[:show_len, 0], mode="lines", line=dict(color="#00FFCC", width=1.5), row=1, col=1)
fig_stereo.add_scatter(x=t_stereo[:show_len]*1e3, y=stereo_audio[:show_len, 1], mode="lines", line=dict(color="#FFA500", width=1.5), row=2, col=1)

fig_stereo.update_layout(template="plotly_dark", height=450, showlegend=False)
fig_stereo.update_xaxes(title="Time (ms)", row=2, col=1)
fig_stereo.update_yaxes(title="Amp", row=1, col=1)
fig_stereo.update_yaxes(title="Amp", row=2, col=1)
fig_stereo.show()

print("🔊 Listen to Stereo Recording (Left = Mic 1, Right = Mic 2):")
ol.play_audio(custom_data=stereo_audio, sample_rate_hz=ol.sample_rate_hz)

## 7. Clean Hardware Shutdown

In [ ]:
ol.close()
print("🔒 Hardware closed and DMA memory released.")